# 04 — Napredne Neuronske Mreže (PyTorch)
**Cilj:** Istražiti napredne NN tehnike i arhitekture za poboljšanje baseline MLP-a:
- Batch Normalization
- Residual / Skip Connections
- Focal Loss (specijalno dizajniran za imbalanced probleme)
- Learning Rate Scheduling
- Hyperparameter Tuning (Optuna — PyTorch alternativa za Keras Tuner)

> **Framework:** Ovaj notebook koristi **PyTorch** umesto TensorFlow/Keras.

---

## 0. Imports

Pored standardnih biblioteka, uvodimo:
- **`optuna`** — Bayesian hyperparameter optimization framework (PyTorch ekvivalent Keras Tuner-a)
- Svi PyTorch moduli ostaju isti kao u prethodnom notebooku

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print(f'PyTorch: {torch.__version__} | Uređaj: {DEVICE}')

## 1. Učitavanje podataka

Učitavamo iste podatke kao u prethodnom notebooku i kreiramo DataLoader-e. Dodatno kreiramo statičke tenzore koji se direktno prosleđuju modelu tokom evaluacije (bez batcheva).

In [ ]:
X_train       = np.load(f'{PROC}X_train.npy')
X_val         = np.load(f'{PROC}X_val.npy')
X_test        = np.load(f'{PROC}X_test.npy')
y_train       = np.load(f'{PROC}y_train.npy')
y_val         = np.load(f'{PROC}y_val.npy')
y_test        = np.load(f'{PROC}y_test.npy')
X_train_smote = np.load(f'{PROC}X_train_smote.npy')
y_train_smote = np.load(f'{PROC}y_train_smote.npy')

with open(f'{PROC}class_weights.json') as f:
    cw = json.load(f)
pos_weight_val = cw['1'] / cw['0']
POS_WEIGHT = torch.tensor([pos_weight_val], dtype=torch.float32).to(DEVICE)

INPUT_DIM = X_train.shape[1]


def make_loader(X, y, batch_size=256, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


train_loader       = make_loader(X_train, y_train)
train_smote_loader = make_loader(X_train_smote, y_train_smote)

X_val_t  = torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
y_val_t  = torch.tensor(y_val,  dtype=torch.float32).unsqueeze(1).to(DEVICE)

print(f'Input dim: {INPUT_DIM}')

## 2. Focal Loss

Focal Loss je posebno dizajniran za ekstremno nebalansirane probleme. Ideja: standardni BCE daje isti doprinos svim uzorcima, a kod nebalansiranog dataseta to znači da legitimne transakcije "preplave" gradijent.

$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- **$(1-p_t)^\gamma$** — *modulating factor*: smanjuje gubitak za uzorke koje mreža lako klasifikuje (visoka $p_t$). Laki primeri manje doprinose gradijentu, mreža se fokusira na teške slučajeve.
- **$\gamma$ (gamma)** — tipično `2.0`. Veće vrednosti jače potiskuju lake primere.
- **$\alpha$ (alpha)** — balansirajući faktor. Tipično `0.25` za pozitivnu klasu.

**PyTorch implementacija:** Nasledimo `nn.Module` (ne Keras klasu). Primamo logite (ne verovatnoće) — računamo BCE ručno radi numeričke stabilnosti.

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss za imbalanced binary classification (prima logite)."""
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        # Sigmoid verovatnoće iz logita
        probs   = torch.sigmoid(logits)
        # Clipping za numeričku stabilnost
        probs   = probs.clamp(min=1e-7, max=1 - 1e-7)

        # Standardni BCE po uzorku
        bce     = -targets * torch.log(probs) - (1 - targets) * torch.log(1 - probs)

        # p_t: verovatnoća tačne klase
        p_t     = targets * probs + (1 - targets) * (1 - probs)

        # Alpha faktor po uzorku
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)

        # Focal loss
        focal   = alpha_t * (1 - p_t) ** self.gamma * bce
        return focal.mean()


print('✅ FocalLoss definisan')

## 3. Model A — Deep MLP sa Batch Normalization

**Batch Normalization** normalizuje aktivacije između slojeva na nivou mini-batcha. Efekti:
- Brža konvergencija (možemo koristiti veći learning rate)
- Regularizacijski efekat (smanjuje potrebu za velikim Dropout-om)
- Redukuje problem "nestajućih gradijenata"

**Preporučeni redosled slojeva:** `Linear → BatchNorm → ReLU → Dropout`  
(BN pre aktivacije je originalni pristup i najčešće daje bolje rezultate)

**Važno za PyTorch:** `BatchNorm1d` se različito ponaša u `model.train()` i `model.eval()` modu — u eval modu koristi statistike (mean/var) akumulirane tokom treniranja, ne statistike trenutnog batcha.

In [ ]:
class DeepBNMLP(nn.Module):
    """
    Deep MLP sa Batch Normalization i Focal Loss.
    Redosled: Linear -> BN -> ReLU -> Dropout
    """
    def __init__(self, input_dim, hidden_units=(128, 64, 32, 16),
                 dropout_rate=0.3, l2=1e-4):
        super().__init__()
        layers_list = []
        in_f = input_dim
        for units in hidden_units:
            layers_list += [
                nn.Linear(in_f, units),
                nn.BatchNorm1d(units),
                nn.ReLU(),
                nn.Dropout(p=dropout_rate)
            ]
            in_f = units
        self.backbone = nn.Sequential(*layers_list)
        self.output   = nn.Linear(in_f, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.output(self.backbone(x))


model_bn = DeepBNMLP(INPUT_DIM)
print(model_bn)
print(f'Parametri: {sum(p.numel() for p in model_bn.parameters()):,}')

### Trening Deep BN MLP-a

Koristimo Focal Loss umesto BCE, uz `class_weight` pristup (originalni nebalansirani podaci). Ova kombinacija je snažna: Focal Loss se sam bori sa nebalansom, dok `class_weight` daje dodatni signal.

In [ ]:
def train_with_focal(model, train_loader, X_val_t, y_val,
                     gamma=2.0, alpha=0.25, epochs=120, lr=1e-3,
                     patience=12, model_name='model', l2=1e-4):
    """Training loop sa Focal Loss-om."""
    model = model.to(DEVICE)
    criterion = FocalLoss(gamma=gamma, alpha=alpha)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6
    )

    best_val_auc = -1.0
    best_weights = None
    patience_cnt = 0
    train_losses, val_losses = [], []
    train_aucs,   val_aucs   = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, all_probs, all_labels = 0.0, [], []

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * X_b.size(0)
            all_probs.extend(torch.sigmoid(model(X_b)).detach().cpu().numpy().ravel())
            all_labels.extend(y_b.cpu().numpy().ravel())

        train_loss = run_loss / len(train_loader.dataset)
        train_auc  = average_precision_score(all_labels, all_probs)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss   = criterion(val_logits, y_val_t).item()
            val_probs  = torch.sigmoid(val_logits).cpu().numpy().ravel()
        val_auc = average_precision_score(y_val, val_probs)

        train_losses.append(train_loss); val_losses.append(val_loss)
        train_aucs.append(train_auc);   val_aucs.append(val_auc)
        scheduler.step(val_auc)

        if epoch % 10 == 0:
            print(f'Epoha {epoch:3d} | Train Loss: {train_loss:.4f} | Val PR-AUC: {val_auc:.4f}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_weights, f'{MODELS}{model_name}.pt')
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'Early stopping (epoha {epoch}) | Beste Val PR-AUC: {best_val_auc:.4f}')
                break

    model.load_state_dict(best_weights)
    return model, train_losses, val_losses, train_aucs, val_aucs


model_bn, tl_bn, vl_bn, ta_bn, va_bn = train_with_focal(
    model_bn, train_loader, X_val_t, y_val,
    gamma=2.0, alpha=0.25, epochs=120, patience=12, model_name='mlp_bn'
)

## 4. Model B — Residual Network (Skip Connections)

**Residualne (skip) konekcije** su ključna inovacija ResNet arhitekture. Umesto da sloj uči direktnu transformaciju $H(x)$, uči **rezidual** $F(x) = H(x) - x$, pa je izlaz bloka:

$$\text{output} = F(x) + x$$

**Prednosti za duboke mreže:**
- Gradijenti mogu direktno "preskočiti" slojeve → nema nestajućih gradijenata
- Mreža može "naučiti identitet" (ignorisati blok) ako to poboljšava performanse
- Efektivniji trening dubljih mreža

**PyTorch implementacija:** Koristimo Functional API (`nn.functional`) direktno u `forward()` metodi za fleksibilniju kontrolu protoka podataka. Skip konekcija se dodaje sa `+` operatorom u forward passu.

In [ ]:
class ResidualBlock(nn.Module):
    """
    Residualni blok: main path + skip connection.
    Ako se dimenzije razlikuju, koristimo projekcioni sloj za skip konekciju.
    """
    def __init__(self, in_features, out_features, dropout_rate=0.3):
        super().__init__()
        self.main_path = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(out_features, out_features),
            nn.BatchNorm1d(out_features)
        )
        # Projekcija za skip ako se dimenzije razlikuju
        self.skip = (
            nn.Linear(in_features, out_features, bias=False)
            if in_features != out_features else nn.Identity()
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        # Main path + skip konekcija
        out = self.main_path(x) + self.skip(x)
        return self.dropout(self.relu(out))


class ResNetMLP(nn.Module):
    """
    ResNet-inspirisana mreža za tabelarne podatke.
    Ulaz se projektuje na prvu skrivenu dimenziju, zatim prolazi kroz
    niz rezidualnih blokova.
    """
    def __init__(self, input_dim, hidden_units=(128, 64, 32), dropout_rate=0.3):
        super().__init__()
        # Početna projekcija ulaza na prvu dimenziju
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_units[0]),
            nn.BatchNorm1d(hidden_units[0]),
            nn.ReLU()
        )
        # Stacked rezidualni blokovi
        blocks = []
        for in_f, out_f in zip(hidden_units[:-1], hidden_units[1:]):
            blocks.append(ResidualBlock(in_f, out_f, dropout_rate))
        self.blocks = nn.Sequential(*blocks)
        self.output = nn.Linear(hidden_units[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.blocks(x)
        return self.output(x)


model_res = ResNetMLP(INPUT_DIM)
print(model_res)
print(f'Parametri: {sum(p.numel() for p in model_res.parameters()):,}')

In [ ]:
model_res, tl_res, vl_res, ta_res, va_res = train_with_focal(
    model_res, train_smote_loader, X_val_t, y_val,
    gamma=2.0, alpha=0.25, epochs=120, patience=12, model_name='mlp_resnet'
)

## 5. Model C — Optuna Hyperparameter Search

**Optuna** je PyTorch-friendly framework za automatsku optimizaciju hiperparametara. Koristi **Bayesian optimization** (TPE — Tree-structured Parzen Estimator) za efikasnu pretragu prostora hiperparametara.

**Kako funkcioniše:**
1. Definišemo `objective` funkciju koja prima `trial` objekat
2. `trial.suggest_*()` metode predlažu vrednosti hiperparametara
3. Optuna trenira model i vraća metriku (val PR-AUC)
4. Nakon svakog `trial`-a, Optuna ažurira probabilistički model i bira sledeće vrednosti pametno

**Prostor pretrage:** broj slojeva, veličina slojeva, dropout, learning rate, gamma Focal Loss-a, L2 regularizacija.

**Pruning:** Optuna može prekinuti neobećavajuće trial-ove rano (`MedianPruner`) — slično Early Stopping-u, ali na nivou pretrage.

In [ ]:
def objective(trial):
    """Optuna objective funkcija — vraća val PR-AUC."""
    # Predlaganje hiperparametara
    n_layers    = trial.suggest_int('n_layers', 2, 5)
    hidden      = [trial.suggest_int(f'units_{i}', 16, 256, step=16)
                   for i in range(n_layers)]
    dropout     = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    l2          = trial.suggest_float('l2', 1e-5, 1e-3, log=True)
    gamma       = trial.suggest_float('focal_gamma', 1.0, 3.0, step=0.5)
    use_bn      = trial.suggest_categorical('use_bn', [True, False])

    # Dinamička gradnja mreže
    layers_list = []
    in_f = INPUT_DIM
    for units in hidden:
        layers_list.append(nn.Linear(in_f, units))
        if use_bn:
            layers_list.append(nn.BatchNorm1d(units))
        layers_list.append(nn.ReLU())
        layers_list.append(nn.Dropout(p=dropout))
        in_f = units
    layers_list.append(nn.Linear(in_f, 1))

    model = nn.Sequential(*layers_list).to(DEVICE)

    criterion = FocalLoss(gamma=gamma, alpha=0.25)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=l2)

    # Kratki trening za pretragu (50 epoha, early stop na patience=8)
    best_auc, patience_cnt = -1.0, 0
    for epoch in range(50):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(X_b), y_b).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_probs = torch.sigmoid(model(X_val_t)).cpu().numpy().ravel()
        val_auc = average_precision_score(y_val, val_probs)

        # Optuna pruning
        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_auc > best_auc:
            best_auc = val_auc
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= 8:
                break

    return best_auc


print('Pokrećem Optuna Bayesian pretragu (20 trials)...')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5)
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'\n=== BEST HYPERPARAMETERS ===')
print(f'Best Val PR-AUC: {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Rekonstruišemo i treniramo best model do kraja
bp = study.best_params
hidden = [bp[f'units_{i}'] for i in range(bp['n_layers'])]

layers_list = []
in_f = INPUT_DIM
for units in hidden:
    layers_list.append(nn.Linear(in_f, units))
    if bp.get('use_bn', False):
        layers_list.append(nn.BatchNorm1d(units))
    layers_list.append(nn.ReLU())
    layers_list.append(nn.Dropout(p=bp['dropout']))
    in_f = units
layers_list.append(nn.Linear(in_f, 1))

model_tuned = nn.Sequential(*layers_list)
print(f'Tuned model arhitektura: {model_tuned}')

model_tuned, tl_tn, vl_tn, ta_tn, va_tn = train_with_focal(
    model_tuned, train_loader, X_val_t, y_val,
    gamma=bp.get('focal_gamma', 2.0),
    alpha=0.25, epochs=150, patience=15,
    lr=bp['lr'], model_name='mlp_tuned', l2=bp['l2']
)

## 6. Helper: evaluacija i poređenje

Definišemo helper funkcije za uniformnu evaluaciju sva tri modela, a zatim vizualizujemo i poredimo rezultate.

In [ ]:
def predict_proba(model, X_tensor):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X_tensor)).cpu().numpy().ravel()


def find_best_threshold(model, X_tensor, y_true):
    y_prob = predict_proba(model, X_tensor)
    thresholds = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1s)]
    print(f'Optimalni threshold: {best_t:.2f} → F1 = {max(f1s):.4f}')
    return best_t


def quick_eval(model, X_test_t, y_test, title, X_val_t, y_val):
    t      = find_best_threshold(model, X_val_t, y_val)
    y_prob = predict_proba(model, X_test_t)
    y_pred = (y_prob >= t).astype(int)
    return {
        'title':     title,
        'roc_auc':   roc_auc_score(y_test, y_prob),
        'pr_auc':    average_precision_score(y_test, y_prob),
        'f1':        f1_score(y_test, y_pred),
        'threshold': t
    }


results = [
    quick_eval(model_bn,    X_test_t, y_test, 'Deep BN MLP + Focal', X_val_t, y_val),
    quick_eval(model_res,   X_test_t, y_test, 'ResNet MLP + Focal',  X_val_t, y_val),
    quick_eval(model_tuned, X_test_t, y_test, 'Tuned MLP (Optuna)',  X_val_t, y_val),
]

results_df = pd.DataFrame(results).set_index('title')
print('\n=== POREĐENJE NAPREDNIH MODELA ===')
display(results_df.style.highlight_max(color='#90EE90', subset=['roc_auc', 'pr_auc', 'f1']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, color in zip(axes, ['roc_auc', 'pr_auc', 'f1'],
                              ['steelblue', 'coral', 'seagreen']):
    bars = ax.barh(results_df.index, results_df[metric],
                   color=color, edgecolor='black', alpha=0.85)
    ax.set_title(metric.upper(), fontweight='bold')
    ax.set_xlim(0.7, 1.0)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Napredni NN Modeli — Poređenje Metrika', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}04_advanced_comparison.png', bbox_inches='tight')
plt.show()

results_df.to_csv(f'{RESULTS}04_advanced_results.csv')
print('\n→ Sledeći notebook: 05_autoencoder.ipynb')